# 📊 Exploratory Data Analysis — UCI Online Retail II

This notebook demonstrates the complete data journey: **raw Excel → cleaned transactions → engineered features**.

We explore data quality issues, revenue patterns, customer behavior distributions, and prepare the foundation for CLV modeling.

In [ ]:
import sys
sys.path.insert(0, 'd:/ML_Project')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.loader import load_raw_data
from src.data.preprocessor import DataPreprocessor
from src.config import *

import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully')

## 1. Load Raw Data

The UCI Online Retail II dataset contains **1M+ transactions** from a UK-based online retailer (Dec 2009 – Dec 2011).

In [ ]:
df_raw = load_raw_data()
print(f'Raw dataset shape: {df_raw.shape}')
print(f'Date range: {df_raw["InvoiceDate"].min()} to {df_raw["InvoiceDate"].max()}')
df_raw.head(10)

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe()

## 2. Missing Values Analysis

Missing `Customer ID` values are the biggest data quality issue — these transactions can’t be attributed to any customer and must be dropped.

In [ ]:
null_counts = df_raw.isnull().sum()
null_pct = (null_counts / len(df_raw) * 100).round(1)

null_df = pd.DataFrame({'Missing Count': null_counts, 'Missing %': null_pct})
null_df = null_df[null_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

fig = px.bar(null_df, x=null_df.index, y='Missing Count',
             text='Missing %', title='Missing Values by Column',
             color='Missing Count', color_continuous_scale='Reds')
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.update_layout(xaxis_title='Column', yaxis_title='Count')
fig.show()

## 3. Data Cleaning Pipeline

Our 7-step cleaning pipeline handles:
1. Drop rows with missing `Customer ID`
2. Remove cancelled orders (Invoice starting with 'C')
3. Filter out non-product stock codes (POST, DOT, BANK CHARGES, etc.)
4. Remove zero/negative quantity and price
5. Cap outliers at 99.5th percentile
6. Convert GBP → INR (×105)
7. Compute `TotalPrice = Quantity × Price`

In [ ]:
prep = DataPreprocessor(convert_to_inr=True)
clean_df = prep.clean(df_raw)

print(f'Before cleaning: {len(df_raw):,} rows')
print(f'After cleaning:  {len(clean_df):,} rows ({len(clean_df)/len(df_raw)*100:.1f}% retained)')
print(f'Unique customers: {clean_df[COL_CUSTOMER].nunique():,}')
print(f'Unique products: {clean_df[COL_STOCK_CODE].nunique():,}')

## 4. Revenue Distribution

Revenue follows a classic **long-tail (Pareto) distribution** — a small fraction of transactions generate the bulk of revenue.

In [ ]:
fig = px.histogram(clean_df, x=COL_TOTAL_PRICE, nbins=200,
                   title='Transaction Revenue Distribution (INR)',
                   labels={COL_TOTAL_PRICE: 'Revenue per Transaction (INR)'})
fig.update_xaxes(range=[0, clean_df[COL_TOTAL_PRICE].quantile(0.95)])
fig.update_layout(yaxis_title='Frequency')
fig.show()

## 5. Geographic Revenue Analysis

In [ ]:
country_rev = clean_df.groupby(COL_COUNTRY)[COL_TOTAL_PRICE].sum().sort_values(ascending=False).head(10)
fig = px.bar(x=country_rev.index, y=country_rev.values,
             title='Top 10 Countries by Total Revenue',
             labels={'x': 'Country', 'y': 'Total Revenue (INR)'},
             color=country_rev.values, color_continuous_scale='Viridis')
fig.show()

## 6. Temporal Patterns

In [ ]:
clean_df['Month'] = pd.to_datetime(clean_df[COL_DATE]).dt.to_period('M').astype(str)
monthly = clean_df.groupby('Month').agg(
    transactions=(COL_INVOICE, 'nunique'),
    revenue=(COL_TOTAL_PRICE, 'sum'),
    customers=(COL_CUSTOMER, 'nunique')
).reset_index()

fig = px.line(monthly, x='Month', y='transactions',
              title='Monthly Transaction Volume',
              labels={'transactions': 'Unique Invoices'})
fig.show()

## 7. Feature Engineering (RFM + Behavioral)

16 features are engineered per customer:

In [ ]:
from src.data.feature_engineering import FeatureEngineer

fe = FeatureEngineer()
features = fe.build_all_features(clean_df)
print(f'Feature matrix: {features.shape[0]} customers x {features.shape[1]} features')
print(f'\nFeatures: {list(features.columns)}')
features.head()

In [ ]:
plt.figure(figsize=(14, 10))
corr = features.select_dtypes(include=[np.number]).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, square=True)
plt.title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

## Summary

| Metric | Value |
|--------|-------|
| Raw transactions | 1,067,371 |
| After cleaning | ~775K (72.7%) |
| Unique customers | ~4,244 (observation window) |
| Engineered features | 16 per customer |
| Currency | GBP → INR (×105) |

**Key Insight**: Revenue is highly concentrated — this Pareto distribution is exactly what makes CLV modeling so valuable. The top 20% of customers likely drive 80%+ of revenue.